# Gradient Boosting

In [5]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report


df = pd.read_csv("pca_features.csv")

TARGET = "chronic_absent"

## 1. Feature Groups

In [6]:
BASE_CAT = [
    "Gen", "Eth", "Fluency", "SpEd",
    "SiteName", "Zip", "Grade", "school_stage",
    "SED"
]
BASE_NUM = [
    "age", "CurrWeightedTotGPA", "Susp", "DaysEnr",
    "year",
]

LEAKAGE = ["DaysAbs", "AttRate", "absence_rate", "gpa_absence"]

# socio / crime（raw features）

RAW_CRIME = [
    "total_crimes", "violent_crimes", "property_crimes", "drug_crimes", "other_crimes",
    "crime_rate", "violent_crime_rate", "property_crime_rate", "drug_crime_rate", "other_crime_rate",
]
RAW_SOCIO = [
    "total_population", "poverty_rate_pct", "median_household_income", "unemployment_rate_pct",
    "high_school_plus_rate_pct", "college_degree_rate_pct",
    "median_gross_rent", "median_home_value", "uninsured_rate_pct",
    "poverty_crime",
]

BASE_NUM_CLEAN = [c for c in BASE_NUM if c not in LEAKAGE]
BASE_CAT_CLEAN = [c for c in BASE_CAT if c not in LEAKAGE]
PCA_IDX = ["socio_index", "crime_index"]

GROUPS = {
    "raw_only": {
        "num": BASE_NUM_CLEAN + RAW_SOCIO + RAW_CRIME,
        "cat": BASE_CAT_CLEAN,
    },
    "pca_only": {
        "num": BASE_NUM_CLEAN + PCA_IDX,
        "cat": BASE_CAT_CLEAN,
    },
    "both": {
        "num": BASE_NUM_CLEAN + RAW_SOCIO + RAW_CRIME + PCA_IDX,
        "cat": BASE_CAT_CLEAN,
    },
}

## 2. Temporal Split

In [7]:
df = df.dropna(subset=[TARGET]).copy()
cutoff = 2122
train_df = df[df["year"] <= cutoff].copy()
test_df  = df[df["year"] > cutoff].copy()

## 3. Experiments

In [10]:
import experiments 
GROUPS = {
    "raw_only": RAW_SOCIO + RAW_CRIME,
    "pca_only": PCA_IDX,
    "both": RAW_SOCIO + RAW_CRIME + PCA_IDX,
}

rows = []
for name, community_cols in GROUPS.items():
    m = experiments.run_gb_experiment(
        train=train_df,
        test=test_df,
        target=TARGET,
        base_num=BASE_NUM,
        base_cat=BASE_CAT,
        community_cols=community_cols,
        leakage=[],
    )
    rows.append({"group": name, **m})

results = pd.DataFrame(rows)
results[["group","auc_roc","auc_pr","precision","recall","f1","recall_at_top_k"]]

,group,auc_roc,auc_pr,precision,recall,f1,recall_at_top_k
0,raw_only,0.727253,0.669819,0.642118,0.525583,0.578036,0.186868
1,pca_only,0.726804,0.669834,0.648205,0.507284,0.569151,0.189020
2,both,0.727731,0.669279,0.643112,0.523466,0.577154,0.187011
